# MOTIF run explorer

Select a result directory or a monitoring directory (`runs/<run-id>`). Inspect each stage’s metrics and search the **full saved lists**, including all candidate losses—not only the top candidates. This notebook only reads files. It does not launch models, rescore verdicts, or change run artifacts.

Install from the repository root: `python -m pip install -e ".[notebook]"`, then `jupyter lab notebooks/run_explorer.ipynb`. Run all cells. On GitHub the saved notebook is an unexecuted preview; open it in Jupyter for interactive controls.

In [ ]:
from pathlib import Path
import sys

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/llm_backdoor_detection/run_explorer.py").exists()), None)
if REPO is None:
    raise RuntimeError("Open this notebook from inside the cloned repository.")
sys.path.insert(0, str(REPO / "src"))
from IPython.display import display
import pandas as pd
from llm_backdoor_detection.run_explorer import (
    discover, browse_stage, read_artifact, list_records, metric_rows, record_table, filter_table
)


## 1. Choose a run directory

Use an absolute path for external runs. The bundled Qwen example has leakage, motifs, search results, and monitoring records. A monitor directory may point to shared output folders; select the exact artifact in each stage instead of assuming the newest file belongs to the run.

`PATH_MAP` relocates historical absolute paths. The bundled `research-results` automatically maps `/home/eliran/motif/results` to the snapshot, even if the original directory exists. For other copies, set your own old-root → new-root mapping. After changing files or the run directory, rerun from this cell to refresh.

In [ ]:
RUN_DIR = REPO / "research-results/qwen/our_model"
# RUN_DIR = Path("/path/to/results/model/runs/20261022T104426Z-dd622951")
PATH_MAP = {}
# PATH_MAP = {"/old/machine/results": "/new/location/results"}

inventory, notes = discover(RUN_DIR, REPO, PATH_MAP)
print(f"{len(inventory)} artifacts discovered under or referenced by {RUN_DIR}")
for note in notes:
    print("NOTE:", note)
display(inventory)


## 2. Run records

Select `status.json` for overall status and elapsed time; choose its `$.stages` list for step status, return codes, commands, and recorded environment overrides. `config.json` holds the configuration; JSONL events and logs are also searchable lists. Pending, failed, or incomplete runs may have missing outputs.

In [ ]:
browse_stage(inventory, "Run records")


## 3. Leakage

Select the summary for generation counts and token-analysis metrics/lists. Select a CSV for every recorded probe, decoding strategy, seed, parameters, and output. Search output text literally, or choose a numeric column and limits. Empty or failed outputs remain visible as recorded. The summary may contain full token ID lists under `token_analysis`.

In [ ]:
browse_stage(inventory, "1 · Leakage")


## 4. Motif extraction

Search cluster sizes, output counts, retention settings, and success status. Browse `$.clusters`, `$.clusters_summary`, `$.selected_cluster.motifs`, unique token IDs, and decoded token lists when present. Choose a cluster record and **Inspect full record** to read its nested motif/example lists. Some older artifacts contain fewer fields.

In [ ]:
browse_stage(inventory, "2 · Motif extraction")


## 5. Search and evaluation

Search metrics such as `verdict`, `baseline`, `asr`, `similarity`, `total_candidates`, and `attention_coverage`. Saved lists include `$.all_candidate_losses`, `$.top_candidates`, `$.evaluation_prompts`, and baseline/known-trigger responses. Sort candidate loss ascending or similarity descending. Inspect a candidate’s original record index to see its full token IDs, raw responses, normalized responses, and similarity metrics.

ASR may be null/unavailable when no target phrase is configured. Do not interpret it as zero. Known-trigger controls are diagnostics; historical settings may differ from current blind defaults. The notebook displays saved verdicts without recomputing them.

In [ ]:
browse_stage(inventory, "3 · Search / evaluation")


## 6. Programmatic queries (optional)

The same functions work without widgets. Select an exact file below, then filter any recorded list. `record_index` always refers to its position in the source list. Limits only affect display; all loaded records remain available. To inspect nested lists, index the source dictionary directly, e.g. `artifact["top_candidates"][0]["responses"]`.

In [ ]:
ARTIFACT_PATH = None  # e.g. Path("/path/to/search_motifs_YYYYMMDD_HHMMSS.json")
LIST_PATH = "$.all_candidate_losses"
TEXT_QUERY = ""
SORT_COLUMN = "loss"

if ARTIFACT_PATH is not None:
    artifact = read_artifact(Path(ARTIFACT_PATH))
    saved_lists = list_records(artifact)
    print({key: len(value) for key, value in saved_lists.items()})
    table = record_table(saved_lists[LIST_PATH])
    matches = filter_table(table, query=TEXT_QUERY, sort_by=SORT_COLUMN)
    print(f"{len(matches)} matching records")
    display(matches.head(50))
else:
    print("Set ARTIFACT_PATH to enable a programmatic query.")
